# SimpleFold ensemble: does averaging beat selecting?

Re-runs the `sf-select` question on the **real** benchmark instead of the old 40-target dev set.

Prior result (40 dev targets, ESMFold era): every blind *selection* rule failed its pre-registered
bar and had within-target rank-rho ~ 0 -- it never ranked samples by true error. The only thing
that worked was the **centroid**: average the samples, no selection needed (2.601 vs 2.875, p=3.6e-8).

Three questions here, on 33 targets stratified by SimpleFold's known error:
1. does the centroid gain (~0.3 A) transfer to the proper benchmark?
2. does ensemble **spread** predict error -- a confidence signal for a model that has none?
3. does medoid selection add anything over the centroid? (expected: no)

In [ ]:
import os, sys, json, time, glob, subprocess, shutil, itertools
import numpy as np

PICKS = ['6ZXND', '7C8VA', '7L6VD', '7M1HD', '7N9VF', '7OMNC', '7PQQB2', '7R1ZC', '7VFAD', '7WHIG', '8ELOB', '8FIFB', '8GZ3N', '8H5UB', '8IM1H', '8OMUB', '8P8JZ', '8Q78C', '8Q94C', '8Q95B', '8SFZC', '8TZUC', '8UW9B', '8XUMC', '9BIUB', '9FC2B', '9G7KG', '9HVIH', '9HZKA', '9M6OB', '9MTEB', '9MTXA', '9XZXD']
NS, STEPS, TAU, SEED = '6', '500', '0.01', '12345'
WORK, SCRATCH = '/kaggle/working', '/tmp/sf'
os.makedirs(SCRATCH, exist_ok=True)
T0 = time.time()
R = {'model':'simplefold_100M','n_samples':int(NS),'tau':float(TAU),'steps':int(STEPS),'seed':int(SEED)}

import torch
n, cap = torch.cuda.device_count(), torch.cuda.get_device_capability(0)
assert n == 2 and cap >= (7,0), (n, cap, torch.cuda.get_device_name(0))
R['gpus'] = [torch.cuda.get_device_name(g) for g in range(n)]
print('GPUs:', R['gpus'])

_ds = glob.glob('/kaggle/input/**/targets_vhh.json', recursive=True); assert len(_ds)==1
TG = {f"{t['pdb']}{t['H']}": t for t in json.load(open(_ds[0]))}
TG = {k: TG[k] for k in PICKS}
assert len(TG) == 33, len(TG)
print(f'{len(TG)} targets')

def must(c, w):
    p = subprocess.run(c, shell=True, text=True, capture_output=True)
    if p.returncode: raise RuntimeError(f'{w}:\n{(p.stdout+p.stderr)[-1500:]}')
must('git clone -q --depth 1 -b fix-multichain-fasta-parsing '
     'https://github.com/omar-A-hassan/ml-simplefold ' + SCRATCH + '/repo', 'clone')
must(f'{sys.executable} -m pip install -q -e {SCRATCH}/repo', 'install')
must(f'{sys.executable} -m pip install -q git+https://github.com/facebookresearch/esm.git', 'esm')
must(f'{sys.executable} -m pip install -q biotite', 'biotite')
INF = f'{SCRATCH}/repo/src/simplefold/inference.py'; s = open(INF).read()
a = '        esm_model = esm_model.to(device)'; assert s.count(a)==1
open(INF,'w').write(s.replace(a, '        esm_model = esm_model.half().to(device)'))
print('fp16 patch applied')

## Fold — 6 samples per target

In [ ]:
fadir = f'{SCRATCH}/fa'; os.makedirs(fadir, exist_ok=True)
for tid, t in TG.items():
    open(f'{fadir}/{tid}.fasta','w').write(f">A|{tid}\n{t['hseq']}\n")

w = subprocess.run([sys.executable,'-c',
    "import torch,esm; torch.hub.load('facebookresearch/esm:main','esm2_t36_3B_UR50D')"],
    capture_output=True, text=True)
if w.returncode: raise RuntimeError('warm failed:\n'+(w.stdout+w.stderr)[-1500:])
print('warm ok')

tids = sorted(TG)
for g in (0,1):
    d = f'{SCRATCH}/sh{g}'; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
for i, tid in enumerate(tids):
    os.link(f'{fadir}/{tid}.fasta', f'{SCRATCH}/sh{i%2}/{tid}.fasta')

t0 = time.time(); procs=[]
for g in (0,1):
    if g: time.sleep(60)
    procs.append(subprocess.Popen(
        ['simplefold','--simplefold_model','simplefold_100M','--backend','torch',
         '--ckpt_dir',f'{SCRATCH}/artifacts','--num_steps',STEPS,'--tau',TAU,
         '--nsample_per_protein',NS,'--seed',SEED,
         '--fasta_path',f'{SCRATCH}/sh{g}','--output_dir',f'{SCRATCH}/out{g}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        env={**os.environ,'CUDA_VISIBLE_DEVICES':str(g)}))
    print('launched', g, flush=True)
for g,p in enumerate(procs):
    o = p.communicate()[0]
    R.setdefault('rc',{})[g] = p.returncode
    if p.returncode: print(f'GPU{g} FAILED', o[-1500:])
R['fold_min'] = round((time.time()-t0)/60,1)
cif = glob.glob(f'{SCRATCH}/out*/**/*.cif', recursive=True)
R['n_cif'] = len(cif)
print(f"{len(cif)} structures ({len(TG)} targets x {NS}) in {R['fold_min']} min")
assert len(cif) >= 0.95*len(TG)*int(NS), (len(cif), len(TG)*int(NS))

## Score

`sample_0` = the single-sample baseline. `centroid` = coordinate average of the 6 after
framework alignment. `medoid` = the sample nearest the others. `oracle` = best of 6 (upper bound).
`spread` = mean CDR-H3 RMSF across the 6 -- the candidate confidence signal.

In [ ]:
import biotite.structure.io.pdbx as pdbx
def kabsch(P,Q):
    Pm,Qm=P.mean(0),Q.mean(0)
    V,_,W=np.linalg.svd((P-Pm).T@(Q-Qm)); D=np.diag([1,1,np.sign(np.linalg.det(V@W))])
    return V@D@W,Pm,Qm
rmsd=lambda a,b: float(np.sqrt(((a-b)**2).sum(-1).mean()))
AT=['N','CA','C','CB']

rows=[]
for tid,t in TG.items():
    g=sorted(glob.glob(f'{SCRATCH}/out*/**/*{tid}*.cif', recursive=True))
    if len(g)<2: R.setdefault('skip',[]).append(f'{tid}:{len(g)}'); continue
    Ln=len(t['hseq'])
    S=[]
    for f in g:
        a=pdbx.get_structure(pdbx.CIFFile.read(f), model=1)
        u=np.unique(a.res_id)
        if len(np.unique(a.chain_id))!=1 or len(u)!=Ln: break
        P=np.full((Ln,4,3),np.nan); rid=a.res_id-a.res_id.min()
        for k,an in enumerate(AT):
            m=a.atom_name==an; P[rid[m],k]=a.coord[m]
        S.append(P)
    if len(S)<2: R.setdefault('skip',[]).append(f'{tid}:parse'); continue
    ref=np.array([[[np.nan]*3 if v is None else v for v in r] for r in t['hca']],float)
    reg={}
    for k in ('cdr1','cdr2','cdr3'):
        m=np.zeros(Ln,bool); lo,hi=t[k]; m[lo:hi]=True; reg[k]=m
    fw=~(reg['cdr1']|reg['cdr2']|reg['cdr3'])
    c3=reg['cdr3']

    # align every sample onto sample 0 by FRAMEWORK, then the ensemble lives in one frame
    base=S[0]; A=[base]
    for P in S[1:]:
        m=np.isfinite(P[...,0])&np.isfinite(base[...,0])&fw[:,None]
        Rot,Pm,Qm=kabsch(P[m],base[m])
        A.append((P-Pm)@Rot+Qm)
    A=np.stack(A)                                   # [nsample, L, 4, 3]
    # THE OLD BUG: identical seeds gave identical structures and a fake 'ensemble'
    spread_all=float(np.nanmean(np.nanstd(A,axis=0)))
    assert spread_all > 1e-6, 'samples are identical -- ensemble is fake'
    cen=np.nanmean(A,axis=0)
    rmsf=np.nanmean(np.nanstd(A,axis=0),axis=(1,2))  # per residue

    def err(P):
        ok=np.isfinite(ref[...,0])&np.isfinite(P[...,0])
        sel=ok&fw[:,None]
        if sel.sum()<60: return None
        Rot,Pm,Qm=kabsch(P[sel],ref[sel]); al=np.full_like(P,np.nan)
        al[ok]=(P[ok]-Pm)@Rot+Qm
        s2=ok&c3[:,None]
        return rmsd(al[s2],ref[s2]) if s2.sum()>=3 else None
    e=[err(P) for P in A]
    if any(x is None for x in e) or err(cen) is None: R.setdefault('skip',[]).append(f'{tid}:score'); continue
    # medoid: mean CDR-H3 distance to the other samples -- no crystal used
    D=np.zeros((len(A),len(A)))
    for i,j in itertools.combinations(range(len(A)),2):
        m=np.isfinite(A[i][...,0])&np.isfinite(A[j][...,0])&c3[:,None]
        D[i,j]=D[j,i]=rmsd(A[i][m],A[j][m])
    med=int(np.argmin(D.sum(1)))
    rows.append({'tid':tid,'n':len(A),'sample_0':round(e[0],3),
                 'centroid':round(err(cen),3),'medoid':round(e[med],3),
                 'oracle':round(min(e),3),'mean_of_n':round(float(np.mean(e)),3),
                 'spread_cdr3':round(float(np.nanmean(rmsf[c3])),3),
                 'spread_fw':round(float(np.nanmean(rmsf[fw])),3),
                 'cdr3_len':int(t['cdr3'][1]-t['cdr3'][0])})
R['rows']=rows
print(f'scored {len(rows)}/{len(TG)}  skips: {len(R.get("skip",[]))}')
json.dump(R, open(f'{WORK}/ensemble_results.json','w'), indent=1, default=str)

## Answer the three questions

In [ ]:
import numpy as np
rng=np.random.default_rng(0)
def ci(v):
    bs=np.median(v[rng.integers(0,len(v),(4000,len(v)))],axis=1)
    return np.median(v), np.percentile(bs,2.5), np.percentile(bs,97.5)
def wilcoxon(a,b):
    d=a-b; d=d[d!=0]
    if len(d)<6: return None
    r=np.argsort(np.argsort(np.abs(d)))+1.0
    W=min(r[d>0].sum(), r[d<0].sum()); n=len(d)
    mu=n*(n+1)/4; sd=np.sqrt(n*(n+1)*(2*n+1)/24)
    from math import erf, sqrt
    z=(W-mu)/sd; return 2*0.5*(1+erf(-abs(z)/sqrt(2)))
rk=lambda a: np.argsort(np.argsort(a)).astype(float)

s0=np.array([r['sample_0'] for r in rows]); cen=np.array([r['centroid'] for r in rows])
mdd=np.array([r['medoid'] for r in rows]); orc=np.array([r['oracle'] for r in rows])
mn=np.array([r['mean_of_n'] for r in rows]); sp=np.array([r['spread_cdr3'] for r in rows])
print(f"n={len(rows)}   CDR-H3 median, framework-aligned\n")
for nm,v in (('single sample',s0),('mean of 6',mn),('MEDOID (blind pick)',mdd),
             ('CENTROID (average)',cen),('oracle best of 6',orc)):
    m,lo,hi=ci(v); print(f"  {nm:<22}{m:>7.3f}   [{lo:.3f}, {hi:.3f}]")
print(f"\nQ1  centroid vs single sample : {np.median(cen-s0):+.3f} A  Wilcoxon p={wilcoxon(cen,s0):.4g}")
print(f"Q3  medoid  vs centroid       : {np.median(mdd-cen):+.3f} A  Wilcoxon p={wilcoxon(mdd,cen):.4g}")
print(f"    oracle gap available      : {np.median(mn-orc):.3f} A")
print(f"    centroid captures         : {(np.median(mn)-np.median(cen))/(np.median(mn)-np.median(orc))*100:.0f}% of it")
print(f"\nQ2  CONFIDENCE -- does ensemble spread predict error?")
for nm,v in (('spread vs single-sample err', s0),('spread vs centroid err', cen)):
    r=float(np.corrcoef(rk(sp),rk(v))[0,1]); print(f"    {nm:<30} rank-rho {r:+.3f}")
r_len=float(np.corrcoef(rk(sp),rk([r['cdr3_len'] for r in rows]))[0,1])
print(f"    (control) spread vs loop length      rank-rho {r_len:+.3f}")
print(f"\n    comparators on the SAME 33 targets: IgFold and NanoBodyBuilder2 medians")
S={'n':len(rows),'single':float(np.median(s0)),'centroid':float(np.median(cen)),
   'medoid':float(np.median(mdd)),'oracle':float(np.median(orc)),'mean_of_n':float(np.median(mn)),
   'p_centroid_vs_single':wilcoxon(cen,s0),'p_medoid_vs_centroid':wilcoxon(mdd,cen),
   'rho_spread_vs_err':float(np.corrcoef(rk(sp),rk(s0))[0,1]),
   'rho_spread_vs_looplen':r_len}
R['summary']=S
json.dump(R, open(f'{WORK}/ensemble_results.json','w'), indent=1, default=str)
print(json.dumps(S,indent=1))